# Pass 1: Grief & Outrage + Context
LLM coding of grief/outrage in YouTube crisis comments.
Batch-by-batch processing per crisis event.

## Methodology

### Motivation
Sentiment labels (POSITIVE / NEGATIVE / NEUTRAL) quantify tone but collapse
moral emotions. Pass 1 isolates Grief and Outrage specifically, plus the
value/relationship invoked as grounds and whose family is referenced —
the moral core of crisis discourse.

### Approach
Chunked LLM calls over slim `df[["video_id", "text"]]` CSV pasted into the
versioned prompt in `../prompts/pass-01-grief-outrage.md`. Output strictly
JSON: `grief_or_outrage ⊆ {Grief, Outrage}`, `value_or_relationship_invoked`,
`family_referenced[]`. Empty array / "None" when absent.

### Pipeline

| Phase | Step | Description |
|---|---|---|
| **1. Load** | Read micro-sample | Preprocessed CSV → `n_sample=30` rows |
| **2. Preprocess** | Slim + clean | Keep `video_id, text`; `dropna`; `strip` |
| **3. Chunk** | Split rows | `chunk_size=15` → `chunk.to_csv(index=False)` per call |
| **4. Call** | LLM coding | `generate_content`, strip fences, `json.loads`, retry-once |
| **5. Export** | Save + inspect | `sample_pass01_results.jsonl` → merge on `row_id` for QA |

### Cross-Analysis
- *Which videos concentrate outrage vs grief?*
- *What values/relationships ground each emotion?*
- *Whose family is invoked, and where?*

### Input / Output
- **Input:** `01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv`
- **Output:** `03-llm-passes/output/{crisis_name}/sample_pass01_results.jsonl`

## Setup
Import dependencies: `pandas` for data, `dotenv` + `google.genai` for LLM
calls, `json/re/time/pathlib` for chunking, parsing, and checkpointing.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai

## Configuration
Micro-sample selector. Papermill injects `crisis_name` / `phase_name` per run.

In [ ]:
# --- papermill parameters (injected by runner) ---
crisis_name = "ateneo-drowning"  # default; overridden by papermill
phase_name = "04-reckoning"  # default; overridden by papermill
n_sample = 30
chunk_size = 15
sample_seed = 42
model = "gemini-2.5-flash"  # pending model lock
	overwrite = False